# SurgeShield — Data Cleaning & Signal Validation

**OSEMN stage: Scrub (S)**

This notebook is the *reasoning* behind the Scrub stage. The reproducible,
productionized versions live in two scripts in this same folder:

| Script | Role |
|--------|------|
| `clean.py` | raw → processed (drop coords, tidy names, types, nulls, dupes) |
| `validate_dataset.py` | three signal diagnostics |

We **import** those scripts here so the notebook and the production code can
never drift apart — the notebook narrates; the scripts are the source of truth.

The work has two parts:
1. **Clean** the raw flood-risk dataset into a tidy, model-ready file.
2. **Validate** whether the dataset actually contains a learnable signal —
   *before* we trust any model's accuracy. This validation is a core
   deliverable of the project, not an afterthought.

> The notebook's working directory is its own stage folder, so all data paths
> are relative like `../1_data/...`.

## 0. Setup

Because folder names beginning with a digit are not importable packages, we
import the two stage scripts as plain modules from the current directory.

In [1]:
import pandas as pd
import numpy as np

# Both scripts live in this folder; import them as modules.
import clean
import validate_dataset as vd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print('clean.py     ->', clean.RAW_PATH.name, '→', clean.PROCESSED_PATH.name)
print('validate.py  -> diagnostics: correlation, permutation importance, shuffled-label')

clean.py     -> flood_risk_dataset.csv → flood_clean.csv
validate.py  -> diagnostics: correlation, permutation importance, shuffled-label


## 1. Obtain → inspect the raw data

We load the **immutable** raw CSV read-only. The raw file is never modified;
cleaning only ever *reads* from `1_data/raw/` and *writes* to `1_data/processed/`.

In [2]:
df_raw = clean.load_raw()
df_raw.head()

[load] raw shape: 10,000 rows x 14 cols


,Latitude,Longitude,Rainfall (mm),Temperature (°C),Humidity (%),River Discharge (m³/s),Water Level (m),Elevation (m),Land Cover,Soil Type,Population Density,Infrastructure,Historical Floods,Flood Occurred
0,18.861663,78.835584,218.999493,34.144337,43.912963,4236.182888,7.415552,377.465433,Water Body,Clay,7276.742184,1,0,1
1,35.570715,77.654451,55.353599,28.778774,27.585422,2472.585219,8.811019,7330.608875,Forest,Peat,6897.736956,0,1,0
2,29.227824,73.108463,103.991908,43.934956,30.108738,977.328053,4.631799,2205.873488,Agricultural,Loam,4361.518494,1,1,1
3,25.361096,85.610733,198.984191,21.569354,34.453690,3683.208933,2.891787,2512.277800,Desert,Sandy,6163.069701,1,1,0
4,12.524541,81.822101,144.626803,32.635692,36.292267,2093.390678,3.188466,2001.818223,Agricultural,Loam,6167.964591,1,0,0


In [3]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Latitude                10000 non-null  float64
 1   Longitude               10000 non-null  float64
 2   Rainfall (mm)           10000 non-null  float64
 3   Temperature (°C)        10000 non-null  float64
 4   Humidity (%)            10000 non-null  float64
 5   River Discharge (m³/s)  10000 non-null  float64
 6   Water Level (m)         10000 non-null  float64
 7   Elevation (m)           10000 non-null  float64
 8   Land Cover              10000 non-null  object 
 9   Soil Type               10000 non-null  object 
 10  Population Density      10000 non-null  float64
 11  Infrastructure          10000 non-null  int64  
 12  Historical Floods       10000 non-null  int64  
 13  Flood Occurred          10000 non-null  int64  
dtypes: float64(9), int64(3), object(2)
memo

### Data-quality snapshot

We check the three things cleaning must address: missing values, duplicate
rows, and the target balance. The raw column names also carry units
(`Rainfall (mm)`, `Temperature (°C)`, `River Discharge (m³/s)`) which we will
tidy. Note that `Latitude`/`Longitude` are present but will be **dropped** from
model features (they are kept upstream only for the map view).

In [4]:
print('Missing values per column:')
print(df_raw.isnull().sum())
print('\nDuplicate rows:', df_raw.duplicated().sum())
print('\nTarget balance (Flood Occurred):')
print(df_raw['Flood Occurred'].value_counts(normalize=True).round(4))

Missing values per column:
Latitude                  0
Longitude                 0
Rainfall (mm)             0
Temperature (°C)          0
Humidity (%)              0
River Discharge (m³/s)    0
Water Level (m)           0
Elevation (m)             0
Land Cover                0
Soil Type                 0
Population Density        0
Infrastructure            0
Historical Floods         0
Flood Occurred            0
dtype: int64

Duplicate rows: 0

Target balance (Flood Occurred):
Flood Occurred
1    0.5057
0    0.4943
Name: proportion, dtype: float64


The target is almost perfectly balanced (~50/50). That is convenient for
training, but — as we will see in the validation section — it is also the first
hint that the features may not separate the two classes at all.

## 2. Scrub → clean the data

`clean.clean()` applies the full sequence in order:

1. **Drop** `Latitude` / `Longitude` (not model features).
2. **Rename** columns to tidy keys (strip units) so the whole stack shares one
   vocabulary — pipeline, Flask API, and frontend.
3. **De-duplicate** exact repeated rows.
4. **Coerce types** (numeric / binary / categorical).
5. **Handle nulls** (drop any row left invalid after coercion).

We pass a copy so the raw frame in memory is left intact for comparison.

In [5]:
df_clean = clean.clean(df_raw.copy())
print('\nShape:', df_raw.shape, '->', df_clean.shape)
df_clean.head()

[drop] removed coordinate columns: ['Latitude', 'Longitude']
[rename] tidied column names -> ['Rainfall', 'Temperature', 'Humidity', 'River Discharge', 'Water Level', 'Elevation', 'Land Cover', 'Soil Type', 'Population Density', 'Infrastructure', 'Historical Floods', 'Flood Occurred']
[dedupe] removed 0 duplicate row(s)
[types] coerced numeric, binary, and categorical columns
[nulls] none found

Shape: (10000, 14) -> (10000, 12)


,Rainfall,Temperature,Humidity,River Discharge,Water Level,Elevation,Land Cover,Soil Type,Population Density,Infrastructure,Historical Floods,Flood Occurred
0,218.999493,34.144337,43.912963,4236.182888,7.415552,377.465433,Water Body,Clay,7276.742184,1,0,1
1,55.353599,28.778774,27.585422,2472.585219,8.811019,7330.608875,Forest,Peat,6897.736956,0,1,0
2,103.991908,43.934956,30.108738,977.328053,4.631799,2205.873488,Agricultural,Loam,4361.518494,1,1,1
3,198.984191,21.569354,34.453690,3683.208933,2.891787,2512.277800,Desert,Sandy,6163.069701,1,1,0
4,144.626803,32.635692,36.292267,2093.390678,3.188466,2001.818223,Agricultural,Loam,6167.964591,1,0,0


In [6]:
print('Cleaned dtypes:')
print(df_clean.dtypes)
print('\nFinal columns (11 features + target):')
print(list(df_clean.columns))

Cleaned dtypes:
Rainfall                     float64
Temperature                  float64
Humidity                     float64
River Discharge              float64
Water Level                  float64
Elevation                    float64
Land Cover            string[python]
Soil Type             string[python]
Population Density           float64
Infrastructure                 int32
Historical Floods              int32
Flood Occurred                 int32
dtype: object

Final columns (11 features + target):
['Rainfall', 'Temperature', 'Humidity', 'River Discharge', 'Water Level', 'Elevation', 'Land Cover', 'Soil Type', 'Population Density', 'Infrastructure', 'Historical Floods', 'Flood Occurred']


### Persist the cleaned data

Write to `1_data/processed/flood_clean.csv`. This is the single cleaned file the
rest of the pipeline (EDA, modeling, interpretation) reads from.

In [7]:
clean.save(df_clean)

[save] wrote cleaned data: C:\Users\ghisl\Documents\Builds\SurgeShield\ml-api\1_data\processed\flood_clean.csv
[save] final shape: 10,000 rows x 12 cols


## 3. Validate → does this dataset carry a learnable signal?

This is the methodological heart of the project. A model can report any
accuracy it likes; the real question is whether the **features actually predict
the target**. We answer it with three independent diagnostics, each implemented
in `validate_dataset.py`.

First we load the cleaned data and split it into features `X` and target `y`.

In [8]:
df = vd.load_data()
X, y = vd.split_X_y(df)
print('Features:', X.shape[1], ' Samples:', len(X))
print('Target balance:', dict(y.value_counts().sort_index()))

[data] loaded processed file: flood_clean.csv  (10000, 12)


Features: 11  Samples: 10000
Target balance: {0: 4943, 1: 5057}


### Diagnostic 1 — Feature/target correlation

The simplest check: the linear association of each feature with the target
(categoricals one-hot expanded). Values clustered near zero mean no linear
relationship exists.

In [9]:
corr = vd.diagnostic_correlation(X, y)


DIAGNOSTIC 1 — Feature/target correlation
Linear association between each feature and the target.
Categoricals are one-hot expanded. Values near 0 mean no linear signal.

                         correlation
Humidity                     +0.0278
Land Cover_Water Body        -0.0261
Temperature                  -0.0158
Soil Type_Loam               -0.0147
Historical Floods            +0.0120
Soil Type_Clay               +0.0119
Land Cover_Desert            +0.0104
Soil Type_Silt               -0.0102
Land Cover_Urban             +0.0096
Land Cover_Agricultural      +0.0084
Soil Type_Sandy              +0.0075
Soil Type_Peat               +0.0056
Water Level                  -0.0052
Population Density           -0.0050
Elevation                    -0.0043
Infrastructure               -0.0038
Rainfall                     -0.0022
River Discharge              -0.0021
Land Cover_Forest            -0.0020

  Strongest |correlation| with target: 0.0278
  --> Result is near-zero: consistent wit

### Diagnostic 2 — Permutation importance

Correlation only sees linear effects. Permutation importance is model-based: we
train a Random Forest, then shuffle each feature in turn and measure how much
test ROC-AUC drops. A feature that matters causes a large drop when scrambled;
importance near zero means the model was not relying on it.

In [10]:
imp = vd.diagnostic_permutation_importance(X, y)


DIAGNOSTIC 2 — Permutation importance
Trains a Random Forest, then shuffles each feature in turn and measures
the drop in test ROC-AUC. Importance near 0 means the feature does not
help the model. Captures non-linear effects that correlation misses.



                    importance     std
Rainfall               +0.0073 +0.0062
River Discharge        +0.0043 +0.0041
Temperature            +0.0030 +0.0075
Elevation              +0.0027 +0.0063
Historical Floods      -0.0011 +0.0030
Soil Type              -0.0011 +0.0067
Infrastructure         -0.0012 +0.0054
Humidity               -0.0063 +0.0090
Land Cover             -0.0075 +0.0080
Water Level            -0.0077 +0.0091
Population Density     -0.0093 +0.0045

  Largest mean importance (AUC drop when shuffled): 0.0073
  --> Result is negligible: consistent with NO learnable signal.


### Diagnostic 3 — Shuffled-label control

The decisive test. We train the **same** model twice: once on the real labels,
and once on **randomly permuted** labels (which destroys any signal by
construction). Both are scored on the same real test labels.

If a genuine signal exists, the real-label model should clearly beat the
shuffled-label control. If the two are indistinguishable — and both sit at
~0.50 ROC-AUC — then the features contain no information about the target.

In [11]:
shuffle_res = vd.diagnostic_shuffled_label(X, y)
shuffle_res


DIAGNOSTIC 3 — Shuffled-label control
The decisive test. We train the SAME model twice:
  (A) on the real labels, and
  (B) on randomly permuted labels (signal destroyed by construction).
Both are scored on the SAME real test labels.
If (A) does not clearly beat (B), the features carry no real signal.



  Real-label test ROC-AUC     : 0.4943
  Shuffled-label test ROC-AUC : 0.4967
  Advantage of real over shuffled : -0.0024

  Reference: 0.50 = random guessing. A dataset WITH signal shows the
  real-label AUC well above 0.50 and well above the shuffled control.
  --> Result is no meaningful advantage: consistent with NO learnable signal.


{'real_auc': 0.4942783076752287,
 'shuffled_auc': 0.4967001007121861,
 'gap': -0.0024217930369574425}

## 4. Finding & conclusion

All three diagnostics agree:

- **Correlation:** every feature/target correlation is near zero (strongest
  ≈ 0.03).
- **Permutation importance:** no feature meaningfully changes test AUC when
  shuffled.
- **Shuffled-label control:** the real-label model does **not** beat the
  shuffled-label model — both hover around 0.50 ROC-AUC (random guessing).

**Conclusion: this public flood dataset carries no learnable signal** linking
the 11 features to flood occurrence.

Rather than fabricate metrics, SurgeShield reports this honestly and reframes
the validation procedure itself as a methodological contribution. The narrative
and the numbers feed `cleaning_report.md` and dissertation **Chapter 4
(Findings)** / **Chapter 5 (Recommendations)**. The deliverable is a complete,
deployable prediction system ready to consume any properly-signalled dataset.

The next stage (`3_eda/`) visualizes these same findings — the ~50% flood rate
across every category and the near-zero correlation heatmap.